<a href="https://colab.research.google.com/github/inmira/E-Commerce-Sales-ETL-Analytics-Pipeline/blob/main/E_Commerce_Sales_ETL_Analytics_Pipeline_Rewritten.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist E-Commerce Sales ETL & Analytics Pipeline — corrected version

## Project objective
Prepare clean, analysis-ready Olist tables for Power BI and SQL analysis.

**Important modelling decision:** orders, order items, and payments have different levels of detail. They must not be merged into one flat table and then summed together. An order can contain multiple items and multiple payment records. Joining both detail tables on `order_id` creates a many-to-many multiplication and duplicates monetary values.

This version keeps separate tables:
- `Orders` — one row per order (delivered orders only, matching the original project scope)
- `OrderItems` — one row per order item
- `Payments` — one row per payment record
- `Customers` — customer dimension
- `Products` — product dimension

The project exports these tables separately and creates SQL analyses using joins that preserve the correct grain.

## 1. Imports and data locations

The source URLs below are taken from the original notebook. The files are downloaded into `S_Data`.

In [1]:
import os
import sqlite3
import gdown
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = "S_Data"
os.makedirs(DATA_PATH, exist_ok=True)

file_urls = {
    "orders.csv": "https://drive.google.com/uc?id=1O3IqZTjiAYF7cHFu41gnrUAxHqdbtlhP",
    "products.csv": "https://drive.google.com/uc?id=12F0wLr-yOwQt07wmwwzKQFqvn0wxXoLf",
    "order_items.csv": "https://drive.google.com/uc?id=1c-zjSNrwUR4ec0UVAnIojzY96HPJvpB8",
    "order_payments.csv": "https://drive.google.com/uc?id=1CuZILmyGWICxHsGP_SahpGDGmUdn3WwY",
    "customers.csv": "https://drive.google.com/uc?id=1vB7CYMX1BPiW-eEH_KGyJiIwV1Jzjg7Q",
}

for filename, url in file_urls.items():
    output_path = os.path.join(DATA_PATH, filename)
    if not os.path.exists(output_path):
        print(f"Downloading {filename}...")
        gdown.download(url, output_path, quiet=False)
    else:
        print(f"Using existing file: {output_path}")

orders_raw = pd.read_csv(os.path.join(DATA_PATH, "orders.csv"), encoding="iso-8859-1")
products = pd.read_csv(os.path.join(DATA_PATH, "products.csv"), encoding="iso-8859-1")
items = pd.read_csv(os.path.join(DATA_PATH, "order_items.csv"), encoding="iso-8859-1")
payments = pd.read_csv(os.path.join(DATA_PATH, "order_payments.csv"), encoding="iso-8859-1")
customers = pd.read_csv(os.path.join(DATA_PATH, "customers.csv"), encoding="iso-8859-1")

for name, df in {
    "orders_raw": orders_raw,
    "items": items,
    "payments": payments,
    "customers": customers,
    "products": products,
}.items():
    print(f"{name}: {df.shape}")

Downloading...
From: https://drive.google.com/uc?id=1O3IqZTjiAYF7cHFu41gnrUAxHqdbtlhP
To: /content/S_Data/orders.csv
100%|██████████| 17.7M/17.7M [00:00<00:00, 54.3MB/s]


Downloading...
From: https://drive.google.com/uc?id=12F0wLr-yOwQt07wmwwzKQFqvn0wxXoLf
To: /content/S_Data/products.csv
100%|██████████| 2.38M/2.38M [00:00<00:00, 108MB/s]


Downloading...
From: https://drive.google.com/uc?id=1c-zjSNrwUR4ec0UVAnIojzY96HPJvpB8
To: /content/S_Data/order_items.csv
100%|██████████| 15.4M/15.4M [00:00<00:00, 31.2MB/s]


Downloading...
From: https://drive.google.com/uc?id=1CuZILmyGWICxHsGP_SahpGDGmUdn3WwY
To: /content/S_Data/order_payments.csv
100%|██████████| 5.78M/5.78M [00:00<00:00, 25.3MB/s]


Downloading...
From: https://drive.google.com/uc?id=1vB7CYMX1BPiW-eEH_KGyJiIwV1Jzjg7Q
To: /content/S_Data/customers.csv
100%|██████████| 9.03M/9.03M [00:00<00:00, 24.6MB/s]


orders_raw: (99441, 8)
items: (112650, 7)
payments: (103886, 5)
customers: (99441, 5)
products: (32951, 9)


## 2. Initial data-quality review

Review missing values and duplicate rows before transformations. The checks below report issues; they do not automatically delete records.

In [2]:
dataframes = {
    "orders_raw": orders_raw,
    "items": items,
    "payments": payments,
    "customers": customers,
    "products": products,
}

quality_summary = []
for name, df in dataframes.items():
    quality_summary.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
    })

pd.DataFrame(quality_summary)

,table,rows,columns,duplicate_rows,missing_cells
0,orders_raw,99441,8,0,4908
1,items,112650,7,0,0
2,payments,103886,5,0,0
3,customers,99441,5,0,0
4,products,32951,9,0,2448


In [3]:
missing_summary = pd.concat(
    {
        name: df.isna().sum().rename("missing_count")
        for name, df in dataframes.items()
    },
    axis=1
)
missing_summary

,orders_raw,items,payments,customers,products
order_id,0.00,0.00,0.00,NaN,NaN
customer_id,0.00,NaN,NaN,0.00,NaN
order_status,0.00,NaN,NaN,NaN,NaN
order_purchase_timestamp,0.00,NaN,NaN,NaN,NaN
order_approved_at,160.00,NaN,NaN,NaN,NaN
order_delivered_carrier_date,"1,783.00",NaN,NaN,NaN,NaN
order_delivered_customer_date,"2,965.00",NaN,NaN,NaN,NaN
order_estimated_delivery_date,0.00,NaN,NaN,NaN,NaN
order_item_id,NaN,0.00,NaN,NaN,NaN
product_id,NaN,0.00,NaN,NaN,0.00


## 3. Prepare dimensions and dates

The original project focuses on orders with `order_status == "delivered"`. We retain that scope and use it consistently in the fact tables.

Product descriptive numeric fields are imputed with their medians, and missing product categories are labelled `unknown`, following the original notebook's cleaning approach.

In [4]:
# Keep only delivered orders, as in the original project.
orders = orders_raw.loc[orders_raw["order_status"].eq("delivered")].copy()

# Parse date fields where available.
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in order_date_columns:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")

if "order_purchase_timestamp" in orders.columns:
    orders["purchase_month"] = (
        orders["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
    )

if {"order_purchase_timestamp", "order_delivered_customer_date"}.issubset(orders.columns):
    orders["delivery_days"] = (
        orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400

if {"order_purchase_timestamp", "order_approved_at"}.issubset(orders.columns):
    orders["approval_hours"] = (
        orders["order_approved_at"] - orders["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600

# Clean product dimension as in the original notebook.
products = products.copy()
if "product_category_name" in products.columns:
    products["product_category_name"] = products["product_category_name"].fillna("unknown")

numeric_product_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]
for col in numeric_product_cols:
    if col in products.columns:
        products[col] = pd.to_numeric(products[col], errors="coerce")
        median_value = products[col].median()
        if pd.notna(median_value):
            products[col] = products[col].fillna(median_value)

# Parse item shipping date.
items = items.copy()
if "shipping_limit_date" in items.columns:
    items["shipping_limit_date"] = pd.to_datetime(
        items["shipping_limit_date"], errors="coerce"
    )

# Ensure monetary fields are numeric.
for df, cols in [
    (items, ["price", "freight_value"]),
    (payments, ["payment_value"]),
]:
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

print("Delivered orders:", len(orders))
print("Order items:", len(items))
print("Payment records:", len(payments))

Delivered orders: 96478
Order items: 112650
Payment records: 103886


## 4. Build fact tables without a many-to-many join

Filter item and payment records to the delivered-order population using the order IDs. Do not join `OrderItems` and `Payments` to each other.

The resulting grain is:
- `orders`: one record per order
- `order_items`: one record per order item
- `payments`: one record per payment sequence/record

In [5]:
delivered_order_ids = orders["order_id"].dropna().unique()

order_items = items.loc[items["order_id"].isin(delivered_order_ids)].copy()
payments_delivered = payments.loc[
    payments["order_id"].isin(delivered_order_ids)
].copy()

# Item-level measure: item price plus freight for that item.
order_items["total_item_value"] = (
    order_items["price"].fillna(0) + order_items["freight_value"].fillna(0)
)

# Keep the customer and product dimensions separate.
customers_dim = customers.drop_duplicates(subset=["customer_id"]).copy()
products_dim = products.drop_duplicates(subset=["product_id"]).copy()

# Key uniqueness checks.
assert orders["order_id"].is_unique, "Orders should contain one row per order_id."
assert customers_dim["customer_id"].is_unique, "Customers dimension key is not unique."
assert products_dim["product_id"].is_unique, "Products dimension key is not unique."

print("Orders:", orders.shape)
print("OrderItems:", order_items.shape)
print("Payments:", payments_delivered.shape)
print("Customers:", customers_dim.shape)
print("Products:", products_dim.shape)

Orders: (96478, 11)
OrderItems: (110197, 8)
Payments: (100756, 5)
Customers: (99441, 5)
Products: (32951, 9)


## 5. Validate table grain and relationships

These checks identify duplicate keys and quantify how many item and payment records exist per order. A high count is not automatically an error: it is normal for an order to contain several items or payment records.

In [6]:
grain_checks = pd.DataFrame({
    "table": ["Orders", "OrderItems", "Payments"],
    "rows": [len(orders), len(order_items), len(payments_delivered)],
    "distinct_order_ids": [
        orders["order_id"].nunique(),
        order_items["order_id"].nunique(),
        payments_delivered["order_id"].nunique(),
    ],
    "duplicate_order_id_rows": [
        int(orders["order_id"].duplicated().sum()),
        int(order_items["order_id"].duplicated().sum()),
        int(payments_delivered["order_id"].duplicated().sum()),
    ],
})
grain_checks

,table,rows,distinct_order_ids,duplicate_order_id_rows
0,Orders,96478,96478,0
1,OrderItems,110197,96478,13719
2,Payments,100756,96477,4279


In [7]:
items_per_order = order_items.groupby("order_id").size().rename("items_count")
payments_per_order = (
    payments_delivered.groupby("order_id").size().rename("payments_count")
)

order_detail_counts = pd.concat(
    [items_per_order, payments_per_order], axis=1
).fillna(0).astype(int)

order_detail_counts["would_be_rows_after_items_payments_join"] = (
    order_detail_counts["items_count"] * order_detail_counts["payments_count"]
)

order_detail_counts.sort_values(
    "would_be_rows_after_items_payments_join", ascending=False
).head(10)

,items_count,payments_count,would_be_rows_after_items_payments_join
order_id,,,
895ab968e7bb0d5659d16cd74cd1650c,3,21,63
fedcd9f7ccdc8cba3a18defedd1a5547,2,19,38
ccf804e764ed5650cd8759557269dc13,1,26,26
68986e4324f6a21481df4e6e89abcf01,2,12,24
c6492b842ac190db807c15aff21a7dd6,2,12,24
a3725dfe487d359b5be08cac48b64ec5,6,4,24
465c2e1bee4561cb39e0db8c5993aafc,2,12,24
285c2e15bebd4ac83635ccc563dc71f4,1,22,22
958c6a70e60365b576dd696ad29bbca2,3,7,21


## 6. Create safe analytical tables

For order-level payment analysis, aggregate payment records to one row per order before joining to `Orders`.

For item/category analysis, aggregate item values at item grain and join to `Products`. Do not use payment values in this item-level table.

In [8]:
# One row per order with total payment amount.
payments_by_order = (
    payments_delivered.groupby("order_id", as_index=False)
    .agg(
        order_payment_value=("payment_value", "sum"),
        payment_records=("payment_value", "size"),
    )
)

# One row per order with item and freight totals.
items_by_order = (
    order_items.groupby("order_id", as_index=False)
    .agg(
        item_price_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
        total_item_value=("total_item_value", "sum"),
        item_count=("order_item_id", "count"),
    )
)

orders_analysis = (
    orders
    .merge(payments_by_order, on="order_id", how="left", validate="one_to_one")
    .merge(items_by_order, on="order_id", how="left", validate="one_to_one")
    .merge(customers_dim, on="customer_id", how="left", validate="many_to_one")
)

orders_analysis[["order_payment_value", "payment_records"]] = (
    orders_analysis[["order_payment_value", "payment_records"]].fillna(0)
)

# Item-level analytical table: one row per item record.
items_analysis = order_items.merge(
    products_dim, on="product_id", how="left", validate="many_to_one"
)

# Attach order-level attributes only; do not attach payment detail.
order_attributes = orders[
    ["order_id", "customer_id", "order_purchase_timestamp", "purchase_month", "order_status"]
].copy()

items_analysis = items_analysis.merge(
    order_attributes, on="order_id", how="left", validate="many_to_one"
)

print("Orders analysis:", orders_analysis.shape)
print("Items analysis:", items_analysis.shape)

Orders analysis: (96478, 21)
Items analysis: (110197, 20)


## 7. Reconcile payment totals

The primary payment total for the delivered-order scope is calculated directly from the filtered payment fact table. The order-level aggregated table should produce the same result.

In [9]:
payment_total_from_fact = payments_delivered["payment_value"].sum()
payment_total_from_orders = orders_analysis["order_payment_value"].sum()

reconciliation = pd.DataFrame({
    "calculation": [
        "Payments fact table",
        "Payments aggregated to order and joined to Orders",
    ],
    "total_payment_value": [
        round(payment_total_from_fact, 2),
        round(payment_total_from_orders, 2),
    ],
})

reconciliation["difference_vs_payment_fact"] = (
    reconciliation["total_payment_value"] - round(payment_total_from_fact, 2)
)

reconciliation

,calculation,total_payment_value,difference_vs_payment_fact
0,Payments fact table,"15,422,461.77",0.00
1,Payments aggregated to order and joined to Orders,"15,422,461.77",0.00


In [10]:
assert np.isclose(
    payment_total_from_fact,
    payment_total_from_orders,
    atol=0.01
), "Payment totals do not reconcile. Check keys and filters."

print(f"Delivered-order payment total: {payment_total_from_fact:,.2f}")

Delivered-order payment total: 15,422,461.77


## 8. Export clean tables for Power BI

Load these CSV files into Power BI as separate tables. Create relationships using the keys listed below rather than merging the two fact tables.

Suggested relationships:
- `Orders[order_id]` 1 → many `OrderItems[order_id]`
- `Orders[order_id]` 1 → many `Payments[order_id]`
- `Customers[customer_id]` 1 → many `Orders[customer_id]`
- `Products[product_id]` 1 → many `OrderItems[product_id]`

Use single-direction filtering from dimensions to facts where possible.

In [11]:
exports = {
    "Orders.csv": orders,
    "OrderItems.csv": order_items,
    "Payments.csv": payments_delivered,
    "Customers.csv": customers_dim,
    "Products.csv": products_dim,
    "Orders_Analysis.csv": orders_analysis,
    "Items_Analysis.csv": items_analysis,
}

for filename, df in exports.items():
    df.to_csv(filename, index=False)
    print(f"Exported {filename}: {df.shape[0]:,} rows")

Exported Orders.csv: 96,478 rows
Exported OrderItems.csv: 110,197 rows
Exported Payments.csv: 100,756 rows
Exported Customers.csv: 99,441 rows
Exported Products.csv: 32,951 rows
Exported Orders_Analysis.csv: 96,478 rows
Exported Items_Analysis.csv: 110,197 rows


## 9. SQL database and validated analytical queries

The database stores separate fact and dimension tables. The queries below aggregate payments from `Payments` and product values from `OrderItems`, preventing payment duplication from item-level joins.

In [12]:
engine = create_engine("sqlite:///olist_analytics.db")

sql_tables = {
    "Orders": orders,
    "OrderItems": order_items,
    "Payments": payments_delivered,
    "Customers": customers_dim,
    "Products": products_dim,
    "Orders_Analysis": orders_analysis,
    "Items_Analysis": items_analysis,
}

for table_name, df in sql_tables.items():
    df.to_sql(table_name, engine, if_exists="replace", index=False)

conn = sqlite3.connect("olist_analytics.db")

### 9.1 Total payment value for delivered orders

In [13]:
query_total_payments = '''
SELECT ROUND(SUM(payment_value), 2) AS delivered_payment_total
FROM Payments;
'''
pd.read_sql(query_total_payments, conn)

,delivered_payment_total
0,"15,422,461.77"


### 9.2 Monthly payment trend

In [14]:
query_monthly_payments = '''
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS purchase_month,
    ROUND(SUM(p.payment_value), 2) AS monthly_payment_value
FROM Payments p
JOIN Orders o
    ON p.order_id = o.order_id
GROUP BY strftime('%Y-%m', o.order_purchase_timestamp)
ORDER BY purchase_month;
'''
monthly_payment_trend = pd.read_sql(query_monthly_payments, conn)
monthly_payment_trend.head()

,purchase_month,monthly_payment_value
0,2016-10,"46,566.71"
1,2016-12,19.62
2,2017-01,"127,545.67"
3,2017-02,"271,298.65"
4,2017-03,"414,369.39"


### 9.3 Revenue by product category (item value, not payment value)

In [15]:
query_category_value = '''
SELECT
    COALESCE(pr.product_category_name, 'unknown') AS product_category_name,
    ROUND(SUM(oi.price), 2) AS item_price_total,
    ROUND(SUM(oi.freight_value), 2) AS freight_total,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS item_plus_freight_total
FROM OrderItems oi
JOIN Products pr
    ON oi.product_id = pr.product_id
GROUP BY COALESCE(pr.product_category_name, 'unknown')
ORDER BY item_plus_freight_total DESC
LIMIT 10;
'''
pd.read_sql(query_category_value, conn)

,product_category_name,item_price_total,freight_total,item_plus_freight_total
0,beleza_saude,"1,233,131.72","178,957.81","1,412,089.53"
1,relogios_presentes,"1,166,176.98","98,156.14","1,264,333.12"
2,cama_mesa_banho,"1,023,434.76","201,774.50","1,225,209.26"
3,esporte_lazer,"954,852.55","163,404.36","1,118,256.91"
4,informatica_acessorios,"888,724.61","143,999.16","1,032,723.77"
5,moveis_decoracao,"711,927.69","168,402.23","880,329.92"
6,utilidades_domesticas,"615,628.69","142,763.56","758,392.25"
7,cool_stuff,"610,204.10","81,476.79","691,680.89"
8,automotivo,"578,966.65","90,488.10","669,454.75"
9,ferramentas_jardim,"470,495.28","96,650.40","567,145.68"


### 9.4 Payment value by customer state

In [16]:
query_state_payments = '''
SELECT
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(p.payment_value), 2) AS payment_value
FROM Payments p
JOIN Orders o
    ON p.order_id = o.order_id
JOIN Customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY payment_value DESC;
'''
pd.read_sql(query_state_payments, conn)

,customer_state,total_orders,payment_value
0,SP,40500,"5,770,266.19"
1,RJ,12350,"2,055,690.45"
2,MG,11354,"1,819,277.61"
3,RS,5345,"861,802.40"
4,PR,4923,"781,919.55"
5,SC,3546,"595,208.40"
6,BA,3256,"591,270.60"
7,DF,2080,"346,146.17"
8,GO,1957,"334,294.22"
9,ES,1995,"317,682.65"


### 9.5 Payment methods

In [17]:
query_payment_methods = '''
SELECT
    payment_type,
    COUNT(*) AS payment_records,
    COUNT(DISTINCT order_id) AS orders_with_payment,
    ROUND(SUM(payment_value), 2) AS payment_value
FROM Payments
GROUP BY payment_type
ORDER BY payment_value DESC;
'''
pd.read_sql(query_payment_methods, conn)

,payment_type,payment_records,orders_with_payment,payment_value
0,credit_card,74586,74304,"12,101,094.88"
1,boleto,19191,19191,"2,769,932.58"
2,voucher,5493,3679,"343,013.19"
3,debit_card,1486,1485,"208,421.12"
